# 全項目進度審計與接續施工
保存實際審計與分流程式；全項目掃描涉及大量磁碟讀取，原腳本禁止刷新既有基線。重跑應另立批次，本筆記本供核查方法審閲。

In [ ]:
from pathlib import Path
__file__ = str(Path.cwd() / '審計.py')
from pathlib import Path
from collections import Counter
from datetime import datetime,timezone
import csv,json,hashlib,os,subprocess,sqlite3
工作=Path(__file__).resolve().parent;根=工作.parents[2];H=lambda b:hashlib.sha256(b).hexdigest()
表=根/'Reference/tables/94_全項目進度審計_B72_20261012';表.mkdir(parents=True,exist_ok=True)
def git(*a):return subprocess.check_output(['git',*a],cwd=根)
def 讀(p):
    with (根/p).open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
def 寫(n,rows,fields=None):
    with (表/n).open('w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields or list(rows[0]));w.writeheader();w.writerows(rows)
assert not (工作/'基線.json').exists(),'不得刷新基線'
tracked={p for p in git('ls-files','-z').decode().split('\0') if p}
基={'提交':git('rev-parse','HEAD').decode().strip(),'上游提交':git('rev-parse','@{u}').decode().strip(),'初始工作區狀態':git('status','--porcelain').decode(),'既有正文':{p:H((根/p).read_bytes()) for p in tracked if p.endswith(('.md','.qmd')) and (根/p).is_file()}}
(工作/'基線.json').write_text(json.dumps(基,ensure_ascii=False,indent=2),encoding='utf-8')
文件=[];跳=[]
for directory,dirs,files in os.walk(根,followlinks=False):
    dirs[:]=[d for d in dirs if d!='.git' and not (Path(directory)/d).is_symlink()]
    for name in files:
        p=Path(directory)/name
        if p.is_symlink():跳.append({'路徑':p.relative_to(根).as_posix(),'原因':'符號連結不追蹤'});continue
        rel=p.relative_to(根).as_posix()
        if p.is_relative_to(工作) or p.is_relative_to(表):continue
        try:
            before=p.stat();h=hashlib.sha256()
            with p.open('rb') as f:
                for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
            after=p.stat()
            文件.append({'路徑':rel,'位元組':before.st_size,'SHA256':h.hexdigest(),'Git已追蹤':rel in tracked,'掃描期間穩定':(before.st_size,before.st_mtime_ns)==(after.st_size,after.st_mtime_ns)})
        except OSError as e:跳.append({'路徑':rel,'原因':type(e).__name__})
寫('全項目文件雜湊快照.csv',文件);寫('跳過與讀取失敗.csv',跳,['路徑','原因'])
print('完成文件快照',len(文件),flush=True)
金融=讀('Reference/tables/50_global_directory_content_b25_20261008/registry_all_4849_work_progress.csv')
逐國=讀('Reference/tables/50_global_directory_content_b25_20261008/registry_all_country_cumulative_source_progress.csv')
科技=讀('Reference/tables/07_global_technology_iso249_20261006/registry_country_technology_coverage.csv')
神經=讀('Reference/tables/07_global_technology_iso249_20261006/registry_neurotechnology_global.csv')
counts={'金融工作總項':len(金融),'金融已嘗試工作':sum(int(r['attempt_observations'] or 0)>0 for r in 金融),'金融末次HTTP200工作':sum(r['last_http_status']=='200' for r in 金融),'金融完整性狀態':dict(Counter(r['directory_completeness'] for r in 金融)),'金融逐國母表行':len(逐國),'金融上市公司完整性狀態':dict(Counter(r['listed_company_completeness'] for r in 逐國)),'金融券商完整性狀態':dict(Counter(r['broker_completeness'] for r in 逐國)),'科技逐國母表行':len(科技),'科技N0至N7狀態':dict(Counter(r['N0_N7_level'] for r in 科技)),'科技研究完成狀態':dict(Counter(r['research_completion_state'] for r in 科技)),'神經科技登記行':len(神經),'神經科技證據狀態':dict(Counter(r['evidence_state'] for r in 神經))}
html=[]
for r in 文件:
    p=根/r['路徑']
    if p.suffix.lower()=='.html' and '_files' not in '/'.join(p.parts) and (p.parent==根 or p.is_relative_to(根/'Reference')) and not p.with_suffix('.qmd').is_file():
        html.append({'HTML路徑':r['路徑'],'SHA256':r['SHA256'],'處置':'成品候選待人工分類；不直接認定來源遺失','同名QMD存在':False})
寫('無同名源文件的成品候選.csv',html,['HTML路徑','SHA256','處置','同名QMD存在'])
舊=json.loads((根/'reports/2026-10-07/git_publish_and_html_recovery/recovery_manifest.json').read_text(encoding='utf-8'))['records'];恢=[]
for r in 舊:
    p=根/r['html'];q=根/r['qmd']
    恢.append({'HTML路徑':r['html'],'QMD路徑':r['qmd'],'HTML存在':p.is_file(),'QMD存在':q.is_file(),'HTML與當時快照雜湊一致':p.is_file() and H(p.read_bytes())==r['sha256'],'核查邊界':'本輪檢查存在與快照；未重新渲染字節一致性'})
寫('歷史源文件重建成果現況.csv',恢)
近期=[]
for d in sorted((根/'reports').glob('2026-*/*')):
    if any(d.name.endswith('_B'+str(i)) for i in range(64,72)):
        p=d/'交付驗收.json'
        if p.exists():近期.append({'批次目錄':d.relative_to(根).as_posix(),'交付回執存在':True,'回執SHA256':H(p.read_bytes()),'核查結果存在':(d/'核查結果.json').exists(),'狀態':'回執存在不是全項目完成證明'})
寫('近期交付回執核對.csv',近期)
db=根/'Reference/tables/50_global_directory_content_b25_20261008/global_directory_content_b25.sqlite'
with sqlite3.connect(db.as_uri()+'?mode=ro',uri=True) as con:完整=con.execute('PRAGMA quick_check').fetchall()
counts.update({'文件快照行':len(文件),'文件快照不穩定數':sum(not r['掃描期間穩定'] for r in 文件),'文件跳過或失敗數':len(跳),'無同名QMD成品候選':len(html),'歷史重建清單行':len(恢),'歷史重建QMD缺失數':sum(not r['QMD存在'] for r in 恢),'歷史重建HTML較當時快照不同數':sum(not r['HTML與當時快照雜湊一致'] for r in 恢),'近期回執數':len(近期),'金融資料庫快速檢查':完整,'審計UTC':datetime.now(timezone.utc).isoformat(),'整體完成率可計算':False,'理由':'全球所有歷史政治實體、上市公司、券商、貨幣與科技項目的可信總分母尚未核定；不能用批次或來源行數當完成率'})
(工作/'審計結果.json').write_text(json.dumps(counts,ensure_ascii=False,indent=2),encoding='utf-8');print(json.dumps(counts,ensure_ascii=False),flush=True)


In [ ]:
from pathlib import Path
__file__ = str(Path.cwd() / '未關聯來源分流.py')
from pathlib import Path
from collections import defaultdict,Counter
import csv,json
工作=Path(__file__).resolve().parent;根=工作.parents[2]
表=根/'Reference/tables/94_全項目進度審計_B72_20261012'
with (根/'Reference/tables/93_全球貨幣與市場代碼全量續核_B71_20261012/未關聯貨幣來源記錄.csv').open(encoding='utf-8-sig',newline='') as f:原=list(csv.DictReader(f))
組=defaultdict(list)
for r in 原:組[r['來源實體名稱原值']].append(r)
中={'ARAB MONETARY FUND':'阿拉伯貨幣基金','BOLIVIA':'玻利維亞','BURMA':'緬甸（歷史英文名稱）','CZECHOSLOVAKIA':'捷克斯洛伐克','EUROPEAN MONETARY CO-OPERATION FUND (EMCF)':'歐洲貨幣合作基金','EUROPEAN UNION':'歐洲聯盟','GERMAN DEMOCRATIC REPUBLIC':'德意志民主共和國','INTERNATIONAL MONETARY FUND (IMF)':'國際貨幣基金組織','LAO':'寮國','MEMBER COUNTRIES OF THE AFRICAN DEVELOPMENT BANK GROUP':'非洲開發銀行集團成員國','NETHERLANDS ANTILLES':'荷屬安的列斯','SAINT MARTIN':'聖馬丁（具體屬地關聯待核）','SERBIA AND MONTENEGRO':'塞爾維亞與蒙特內哥羅','SISTEMA UNITARIO DE COMPENSACION REGIONAL DE PAGOS "SUCRE"':'蘇克雷區域支付補償體系','SOUTHERN RHODESIA':'南羅德西亞','SWAZILAND':'史瓦濟蘭（歷史英文名稱）','TURKEY':'土耳其','UNION OF SOVIET SOCIALIST REPUBLICS':'蘇維埃社會主義共和國聯盟','VENEZUELA':'委內瑞拉','YEMEN, DEMOCRATIC':'民主葉門','YUGOSLAVIA':'南斯拉夫','ZAIRE':'薩伊（歷史英文名稱）','ZZ01_Bond Markets Unit European_EURCO':'歐洲債券市場計價單位EURCO','ZZ01_Gold-Franc':'金法郎','ZZ02_Bond Markets Unit European_EMU-6':'歐洲債券市場計價單位EMU-6','ZZ02_RINET Funds Code':'RINET（原識別名）基金代碼','ZZ03_Bond Markets Unit European_EUA-9':'歐洲債券市場計價單位EUA-9','ZZ04_Bond Markets Unit European_EUA-17':'歐洲債券市場計價單位EUA-17','ZZ05_UIC-Franc':'國際鐵路聯盟計價法郎','ZZ06_Testing_Code':'測試代碼','ZZ07_No_Currency':'無貨幣','ZZ08_Gold':'黃金','ZZ09_Palladium':'鈀','ZZ10_Platinum':'鉑','ZZ11_Silver':'白銀'}
行=[]
for name,rs in sorted(組.items()):
    clean=name.strip();assert clean in 中
    if clean.startswith('ZZ'):kind='ZZ前綴專用實體名稱；不是缺失國家數'
    elif any(s in clean for s in ['FUND','BANK GROUP','SISTEMA','EUROPEAN UNION']):kind='跨國機構或計价體系候選；角色仍待核'
    elif all(r['狀態中譯']=='歷史' for r in rs):kind='只見於本次歷史來源表；政治類型另核'
    else:kind='名稱與國別關聯待核'
    行.append({'來源實體名稱原值':name,'中文編輯參考':中[clean],'中譯證據':'編輯中譯；不是法定名稱或官方國別對照認證','来源行數':len(rs),'現行來源行':sum(r['狀態中譯']=='現行' for r in rs),'歷史來源行':sum(r['狀態中譯']=='歷史' for r in rs),'分流類別':kind,'記錄識別碼JSON':json.dumps([r['記錄識別碼'] for r in rs],ensure_ascii=False),'代碼與撤回日期JSON':json.dumps([{'字母代碼':r['字母代碼'],'撤回日期原值':r['撤回日期原值']} for r in rs],ensure_ascii=False),'國別關聯已提升':False,'接續驗收條件':'須以適用版本的官方證據確認實體類型及國別／歷史實體關聯；不強行塞入現行母表'})
assert sum(r['来源行數'] for r in 行)==len(原)==42 and len(行)==35
with (表/'未關聯貨幣來源全量分流.csv').open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(行[0]));w.writeheader();w.writerows(行)
(工作/'分流驗收.json').write_text(json.dumps({'原始來源行':len(原),'來源名稱數':len(行),'全量保留與對賬':True,'已增加中文編輯參考':35,'分流類別分布':dict(Counter(r['分流類別'] for r in 行)),'國別關聯已提升':False},ensure_ascii=False,indent=2),encoding='utf-8')
print('42條來源記錄、35個名稱已全量分流，不提升未知國別證據。')


In [ ]:
from pathlib import Path
__file__ = str(Path.cwd() / '驗收.py')
from pathlib import Path
from collections import Counter
import json,csv,hashlib
工作=Path(__file__).resolve().parent;根=工作.parents[2];H=lambda b:hashlib.sha256(b).hexdigest()
基=json.loads((工作/'基線.json').read_text(encoding='utf-8'));附=json.loads((工作/'附加收據.json').read_text(encoding='utf-8'));結=json.loads((工作/'審計結果.json').read_text(encoding='utf-8'))
for p,h in 基['既有正文'].items():
    b=(根/p).read_bytes()
    if p==附['主報告']:
        n=附['原文位元組'];assert H(b[:n])==h==附['原文雜湊'] and H(b[n:])==附['附加雜湊']
    else:assert H(b)==h,'既有正文變更：'+p
表=根/'Reference/tables/94_全項目進度審計_B72_20261012'
def 讀(p):
    with p.open(encoding='utf-8-sig',newline='') as f:return list(csv.DictReader(f))
快=讀(表/'全項目文件雜湊快照.csv');assert len(快)==結['文件快照行'] and len({r['路徑'] for r in 快})==len(快)
assert sum(r['掃描期間穩定']=='False' for r in 快)==結['文件快照不穩定數']==0
assert len(讀(表/'跳過與讀取失敗.csv'))==結['文件跳過或失敗數']==0
index={r['路徑']:r['SHA256'] for r in 快}
inputs=['Reference/tables/50_global_directory_content_b25_20261008/registry_all_4849_work_progress.csv','Reference/tables/50_global_directory_content_b25_20261008/registry_all_country_cumulative_source_progress.csv','Reference/tables/07_global_technology_iso249_20261006/registry_country_technology_coverage.csv','Reference/tables/07_global_technology_iso249_20261006/registry_neurotechnology_global.csv','Reference/tables/93_全球貨幣與市場代碼全量續核_B71_20261012/未關聯貨幣來源記錄.csv']
for p in inputs:assert H((根/p).read_bytes())==index[p],'審計輸入已變更'
f=讀(根/inputs[0]);assert len(f)==結['金融工作總項']==4849
assert sum(int(r['attempt_observations'] or 0)>0 for r in f)==結['金融已嘗試工作']
assert sum(r['last_http_status']=='200' for r in f)==結['金融末次HTTP200工作']
assert dict(Counter(r['directory_completeness'] for r in f))==結['金融完整性狀態']
分=讀(表/'未關聯貨幣來源全量分流.csv');原=讀(根/inputs[-1])
ids=[x for r in 分 for x in json.loads(r['記錄識別碼JSON'])]
assert len(分)==35 and len(ids)==len(原)==42 and Counter(ids)==Counter(r['記錄識別碼'] for r in 原)
assert all(r['國別關聯已提升']=='False' for r in 分)
assert 結['金融資料庫快速檢查']==[['ok']] and not 結['整體完成率可計算']
assert 結['歷史重建清單行']==66 and 結['歷史重建QMD缺失數']==0 and 結['歷史重建HTML較當時快照不同數']==0
html=根/'Reference/全項目進度審計_B72_20261012.html';assert html.stat().st_size>1000
(工作/'交付驗收.json').write_text(json.dumps({'既有正文保留數':len(基['既有正文']),'主報告原文保留':True,'文件快照行':len(快),'審計輸入雜湊核對':True,'金融進度對賬':True,'42項來源全量分流對賬':True,'未提升未知證據':True,'成品雜湊':H(html.read_bytes()),'全項目內容實質核實已完成':False},ensure_ascii=False,indent=2),encoding='utf-8')
print('文件快照、審計來源、進度與分流對賬、正文保留驗收通過。')
